# F1 Race Prediction: Predicting Podium Finishers

**A Machine Learning Project Using 2020-2024 Formula 1 Data**

---

## Table of Contents

1. [Problem Definition & Setup](#1-problem-definition--setup)
2. [Data Cleaning & Feature Engineering](#2-data-cleaning--feature-engineering)
3. [Exploratory Data Analysis](#3-exploratory-data-analysis)
4. [Feature Selection & Preprocessing](#4-feature-selection--preprocessing)
5. [Model Building & Comparison](#5-model-building--comparison)
6. [Best Model Evaluation](#6-best-model-evaluation)
7. [Key Insights](#7-key-insights)
8. [Conclusions](#8-conclusions)


---
## 1. Problem Definition & Setup

### Objective
Predict whether a driver will finish on the **podium** (top 3) in a Formula 1 race.

### Why This Matters
- **Podium finishes** are the most prized results in F1, directly impacting championship standings.
- Understanding what factors contribute to podium finishes helps teams optimize strategy.
- This is a classic **binary classification** problem with real-world sports analytics applications.

### Dataset
- **Source**: Simulated F1 race data based on 2020-2024 seasons (103 Grands Prix, ~2140 entries)
- **Features**: Driver stats, constructor performance, grid position, circuit characteristics, weather
- **Target**: `podium_finish` (1 = finished P1-P3, 0 = otherwise)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import GradientBoostingClassifier

# Plotting settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

# Create output directory
import os
os.makedirs('../outputs/figures', exist_ok=True)

print('All libraries imported successfully!')

### Load Data


In [ ]:
# Load the dataset
df = pd.read_csv('../data/f1_data.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nTotal races: {df.groupby(["season", "race_name"]).ngroups}')
print(f'Seasons covered: {sorted(df["season"].unique())}')
print(f'Unique drivers: {df["driver"].nunique()}')
print(f'Unique constructors: {df["constructor"].nunique()}')

### Data Overview


In [ ]:
# Display first few rows
display(df.head(10))

In [ ]:
# Dataset info
df.info()

In [ ]:
# Statistical summary
display(df.describe())

In [ ]:
# Check data types
print('Data types:')
print(df.dtypes)

---
## 2. Data Cleaning & Feature Engineering

### Missing Values
DNF entries will have missing `race_time_seconds` and `gap_to_winner_seconds`. These are expected and will be handled appropriately.


In [ ]:
# Check for missing values
print('Missing values per column:')
missing = df.isnull().sum()
print(missing[missing > 0])

print(f'\nTotal DNF entries: {df["did_not_finish"].sum()}')
print(f'DNF rate: {df["did_not_finish"].mean():.2%}')

In [ ]:
# Handle missing values for DNF entries
df['race_time_seconds'] = df['race_time_seconds'].fillna(-1)
df['gap_to_winner_seconds'] = df['gap_to_winner_seconds'].fillna(-1)

print('Missing values after handling:')
print(df.isnull().sum().sum())

### Feature Engineering

Let's create several derived features to capture meaningful patterns:

1. **Grid position categories** (front row, top 6, midfield, backmarker)
2. **Constructor performance tier** (top team, midfield, backmarker)
3. **Driver experience score** (career wins + podiums)
4. **Circuit type encoding** (street vs permanent)
5. **Weather encoding** (dry, wet, mixed)
6. **Constructor points per season ratio**

In [ ]:
# Feature 1: Grid position categories
def grid_category(pos):
    if pos <= 2:
        return 'front_row'
    elif pos <= 6:
        return 'top_6'
    elif pos <= 12:
        return 'midfield'
    else:
        return 'backmarker'

df['grid_category'] = df['grid_position'].apply(grid_category)

# Feature 2: Constructor performance tier
def constructor_tier(team):
    top_teams = ['Red Bull Racing', 'Mercedes', 'Ferrari', 'McLaren']
    mid_teams = ['Aston Martin', 'Alpine', 'Williams']
    if team in top_teams:
        return 'top'
    elif team in mid_teams:
        return 'mid'
    else:
        return 'backmarker'

df['constructor_tier'] = df['constructor'].apply(constructor_tier)

print('Grid category distribution:')
print(df['grid_category'].value_counts())
print('\nConstructor tier distribution:')
print(df['constructor_tier'].value_counts())

In [ ]:
# Feature 3: Driver experience score
df['driver_experience'] = df['driver_career_wins_before_race'] + df['driver_career_podiums_before_race']

# Feature 4: Circuit type encoding
df['is_street_circuit'] = (df['circuit_type'] == 'street').astype(int)

# Feature 5: Weather encoding
weather_map = {'dry': 0, 'mixed': 1, 'wet': 2}
df['weather_encoded'] = df['weather'].map(weather_map)

# Feature 6: Constructor points-per-season ratio (normalized)
max_possible_points = df.groupby('season')['constructor_previous_season_points'].transform('max')
df['constructor_points_ratio'] = df['constructor_previous_season_points'] / max_possible_points

print('New features created successfully!')
print(f'New columns: {["grid_category", "constructor_tier", "driver_experience", "is_street_circuit", "weather_encoded", "constructor_points_ratio"]}')

# Verify
display(df[['driver', 'grid_position', 'grid_category', 'constructor', 'constructor_tier', 
             'driver_experience', 'is_street_circuit', 'weather_encoded', 'constructor_points_ratio']].head(10))

In [ ]:
# Label encoding for categorical features
le_grid_cat = LabelEncoder()
le_const_tier = LabelEncoder()

df['grid_category_enc'] = le_grid_cat.fit_transform(df['grid_category'])
df['constructor_tier_enc'] = le_const_tier.fit_transform(df['constructor_tier'])

# Compound encoding
compound_map = {'soft': 2, 'medium': 1, 'hard': 0}
df['compound_encoded'] = df['compound_used'].map(compound_map)

print('Label encoders:')
print(f'  Grid categories: {list(le_grid_cat.classes_)}')
print(f'  Constructor tiers: {list(le_const_tier.classes_)}')

---
## 3. Exploratory Data Analysis

Let's explore the data to understand distributions, relationships, and patterns.


### 3.1 Target Variable Distribution

The target variable `podium_finish` is **imbalanced**, which is expected: only the top 3 finishers in each race achieve a podium.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
ax1 = axes[0]
counts = df['podium_finish'].value_counts()
bars = ax1.bar(['Non-Podium (0)', 'Podium (1)'], counts.values, color=['#3498db', '#e74c3c'], 
               edgecolor='white', linewidth=1.5)
ax1.set_xlabel('Podium Finish', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('Podium vs Non-Podium Distribution', fontsize=14, fontweight='bold')
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
             f'{val} ({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=11)

# Pie chart
ax2 = axes[1]
ax2.pie(counts.values, labels=['Non-Podium', 'Podium'], autopct='%1.1f%%',
        colors=['#3498db', '#e74c3c'], startangle=90, explode=(0.05, 0.05),
        textprops={'fontsize': 12})
ax2.set_title('Class Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nImbalance ratio: {counts[0]/counts[1]:.1f}:1 (non-podium : podium)')

### 3.2 Total Podiums by Constructor

Which teams dominate the podium?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

podium_by_team = df[df['podium_finish'] == 1]['constructor'].value_counts()
colors = sns.color_palette('Set2', n_colors=len(podium_by_team))

bars = ax.barh(podium_by_team.index[::-1], podium_by_team.values[::-1], color=colors[::-1],
               edgecolor='white', linewidth=1)

for bar, val in zip(bars, podium_by_team.values[::-1]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Total Podium Finishes (2020-2024)', fontsize=12)
ax.set_title('Total Podium Finishes by Constructor', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/figures/02_podiums_by_constructor.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Podium Rate by Driver (Top 15)

Which drivers have the highest probability of finishing on the podium?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

driver_stats = df.groupby('driver').agg(
    races=('podium_finish', 'count'),
    podiums=('podium_finish', 'sum')
).reset_index()
driver_stats['podium_rate'] = driver_stats['podiums'] / driver_stats['races']
driver_stats = driver_stats[driver_stats['races'] >= 15]  # minimum races filter
driver_stats = driver_stats.nlargest(15, 'podium_rate')

colors = plt.cm.RdYlGn(np.linspace(0.9, 0.3, len(driver_stats)))
bars = ax.barh(driver_stats['driver'][::-1], driver_stats['podium_rate'][::-1] * 100, 
               color=colors[::-1], edgecolor='white', linewidth=1)

for bar, val, races, pods in zip(bars, driver_stats['podium_rate'][::-1]*100, 
                                   driver_stats['races'][::-1], driver_stats['podiums'][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}% ({pods}/{races})', va='center', fontsize=10)

ax.set_xlabel('Podium Rate (%)', fontsize=12)
ax.set_title('Podium Rate by Driver (Min. 15 Races)', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/figures/03_podium_rate_by_driver.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Grid Position vs Podium

How does starting grid position affect the likelihood of a podium finish?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot
ax1 = axes[0]
podium_data = df[df['podium_finish'] == 1]['grid_position']
non_podium_data = df[df['podium_finish'] == 0]['grid_position']

bp = ax1.boxplot([podium_data, non_podium_data], label=['Podium', 'Non-Podium'],
                 patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('#e74c3c')
bp['boxes'][1].set_facecolor('#3498db')
for box in bp['boxes']:
    box.set_alpha(0.7)

ax1.set_ylabel('Grid Position', fontsize=12)
ax1.set_title('Grid Position: Podium vs Non-Podium', fontsize=14, fontweight='bold')
ax1.invert_yaxis()  # Lower grid position is better

# Podium rate by grid position
ax2 = axes[1]
podium_by_grid = df.groupby('grid_position')['podium_finish'].agg(['mean', 'count'])
podium_by_grid = podium_by_grid[podium_by_grid['count'] >= 10]  # sufficient sample size

ax2.bar(podium_by_grid.index, podium_by_grid['mean'] * 100, 
        color='#2ecc71', alpha=0.8, edgecolor='white', linewidth=1)
ax2.axhline(y=df['podium_finish'].mean() * 100, color='red', linestyle='--', 
            label=f'Overall average ({df["podium_finish"].mean()*100:.1f}%)')
ax2.set_xlabel('Grid Position', fontsize=12)
ax2.set_ylabel('Podium Rate (%)', fontsize=12)
ax2.set_title('Podium Rate by Grid Position', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/figures/04_grid_position_vs_podium.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.5 Podium Rate by Circuit Type

Does the type of circuit (street vs permanent) affect podium probability?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

podium_by_circuit = df.groupby('circuit_type').agg(
    total=('podium_finish', 'count'),
    podiums=('podium_finish', 'sum')
)
podium_by_circuit['rate'] = podium_by_circuit['podiums'] / podium_by_circuit['total']

colors = ['#3498db', '#e67e22']
bars = ax.bar(podium_by_circuit.index, podium_by_circuit['rate'] * 100, 
               color=colors, edgecolor='white', linewidth=1.5, width=0.5)

for bar, rate, total, pods in zip(bars, podium_by_circuit['rate']*100, 
                                     podium_by_circuit['total'], podium_by_circuit['podiums']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{rate:.1f}%\n({pods}/{total} races)', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('Podium Rate (%)', fontsize=12)
ax.set_title('Podium Rate by Circuit Type', fontsize=14, fontweight='bold')
ax.set_ylim(0, 20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/figures/05_podium_rate_by_circuit_type.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.6 Podium Rate by Weather Condition

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

podium_by_weather = df.groupby('weather').agg(
    total=('podium_finish', 'count'),
    podiums=('podium_finish', 'sum')
)
podium_by_weather['rate'] = podium_by_weather['podiums'] / podium_by_weather['total']

weather_colors = {'dry': '#f1c40f', 'mixed': '#95a5a6', 'wet': '#3498db'}
colors = [weather_colors[w] for w in podium_by_weather.index]
bars = ax.bar(podium_by_weather.index, podium_by_weather['rate'] * 100, 
               color=colors, edgecolor='white', linewidth=1.5, width=0.5)

for bar, rate, total, pods in zip(bars, podium_by_weather['rate']*100, 
                                     podium_by_weather['total'], podium_by_weather['podiums']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{rate:.1f}%\n({pods}/{total})', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('Podium Rate (%)', fontsize=12)
ax.set_title('Podium Rate by Weather Condition', fontsize=14, fontweight='bold')
ax.set_ylim(0, 20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/figures/06_podium_rate_by_weather.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.7 Constructor Performance Over Seasons

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Top 6 constructors by total podiums
top_teams = df[df['podium_finish'] == 1]['constructor'].value_counts().nlargest(6).index

for team in top_teams:
    team_data = df[(df['constructor'] == team) & (df['podium_finish'] == 1)]
    season_podiums = team_data.groupby('season')['podium_finish'].sum()
    ax.plot(season_podiums.index, season_podiums.values, marker='o', linewidth=2, 
            markersize=8, label=team)

ax.set_xlabel('Season', fontsize=12)
ax.set_ylabel('Podium Finishes', fontsize=12)
ax.set_title('Constructor Podium Performance Across Seasons', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.set_xticks([2020, 2021, 2022, 2023, 2024])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/figures/07_constructor_performance_seasons.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Feature Selection & Preprocessing

### Strategy
- **Train set**: 2020-2023 seasons
- **Test set**: 2024 season (simulates real-world prediction on unseen data)
- **Class imbalance**: Addressed using class weights (no SMOTE, as we want to evaluate on realistic distributions)


In [ ]:
# Define feature columns
feature_cols = [
    'grid_position',
    'grid_category_enc',
    'constructor_tier_enc',
    'driver_experience',
    'is_street_circuit',
    'weather_encoded',
    'compound_encoded',
    'driver_career_wins_before_race',
    'driver_career_podiums_before_race',
    'constructor_wins_before_season',
    'constructor_previous_season_points',
    'constructor_points_ratio',
    'track_temperature',
    'qualifying_gap_to_pole',
]

X = df[feature_cols]
y = df['podium_finish']

print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape: {y.shape}')
print(f'\nSelected features ({len(feature_cols)}):')
for f in feature_cols:
    print(f'  - {f}')

In [ ]:
# Train-test split by season
train_mask = df['season'] <= 2023
test_mask = df['season'] == 2024

X_train = X[train_mask]
X_test = X[test_mask]
y_train = y[train_mask]
y_test = y[test_mask]

print(f'Training set: {X_train.shape[0]} samples ({len(train_mask[train_mask].index)} races)')
print(f'Test set:     {X_test.shape[0]} samples ({len(test_mask[test_mask].index)} races)')
print(f'\nTraining target distribution:')
print(y_train.value_counts())
print(f'\nTest target distribution:')
print(y_test.value_counts())

In [ ]:
# Scale numerical features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Features scaled successfully!')
print(f'Scaled training set shape: {X_train_scaled.shape}')

# Compute class weights for imbalanced data
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))

print(f'\nClass weights: {class_weight_dict}')

---
## 5. Model Building & Comparison

We train and compare three models:
1. **Logistic Regression** (baseline)
2. **Random Forest** (ensemble)
3. **XGBoost** (gradient boosting)

All models use class weights to handle the imbalanced dataset.

### 5.1 Logistic Regression (Baseline)

In [ ]:
# Logistic Regression with class weights
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    solver='lbfgs'
)

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
print(f'Logistic Regression CV ROC-AUC: {lr_cv_scores.mean():.4f} (+/- {lr_cv_scores.std()*2:.4f})')

# Fit on training data
lr_model.fit(X_train_scaled, y_train)

# Predictions
lr_pred = lr_model.predict(X_test_scaled)
lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

lr_metrics = {
    'Accuracy': accuracy_score(y_test, lr_pred),
    'Precision': precision_score(y_test, lr_pred),
    'Recall': recall_score(y_test, lr_pred),
    'F1 Score': f1_score(y_test, lr_pred),
    'ROC-AUC': roc_auc_score(y_test, lr_proba)
}

print('\nLogistic Regression Test Metrics:')
for metric, value in lr_metrics.items():
    print(f'  {metric}: {value:.4f}')

### 5.2 Random Forest

In [ ]:
# Random Forest with class weights
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Cross-validation
rf_cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
print(f'Random Forest CV ROC-AUC: {rf_cv_scores.mean():.4f} (+/- {rf_cv_scores.std()*2:.4f})')

# Fit on training data
rf_model.fit(X_train_scaled, y_train)

# Predictions
rf_pred = rf_model.predict(X_test_scaled)
rf_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

rf_metrics = {
    'Accuracy': accuracy_score(y_test, rf_pred),
    'Precision': precision_score(y_test, rf_pred),
    'Recall': recall_score(y_test, rf_pred),
    'F1 Score': f1_score(y_test, rf_pred),
    'ROC-AUC': roc_auc_score(y_test, rf_proba)
}

print('\nRandom Forest Test Metrics:')
for metric, value in rf_metrics.items():
    print(f'  {metric}: {value:.4f}')

### 5.3 XGBoost

In [ ]:
# Gradient Boosting for imbalance
# Using sklearn's GradientBoostingClassifier (equivalent to XGBoost)
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()

xgb_model = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

# Compute sample weights for class imbalance
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# Cross-validation (without sample_weight for simplicity, as GBC doesn't easily support it in CV)
xgb_cv_scores = cross_val_score(xgb_model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
print(f'Gradient Boosting CV ROC-AUC: {xgb_cv_scores.mean():.4f} (+/- {xgb_cv_scores.std()*2:.4f})')

# Fit on training data with sample weights for class imbalance
xgb_model.fit(X_train_scaled, y_train, sample_weight=sample_weights)

# Predictions
xgb_pred = xgb_model.predict(X_test_scaled)
xgb_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]

xgb_metrics = {
    'Accuracy': accuracy_score(y_test, xgb_pred),
    'Precision': precision_score(y_test, xgb_pred),
    'Recall': recall_score(y_test, xgb_pred),
    'F1 Score': f1_score(y_test, xgb_pred),
    'ROC-AUC': roc_auc_score(y_test, xgb_proba)
}

print('\nGradient Boosting Test Metrics:')
for metric, value in xgb_metrics.items():
    print(f'  {metric}: {value:.4f}')

### 5.4 Model Comparison

In [ ]:
# Compare all models
metrics_df = pd.DataFrame([
    {'Model': 'Logistic Regression', **lr_metrics},
    {'Model': 'Random Forest', **rf_metrics},
    {'Model': 'XGBoost', **xgb_metrics},
])

print('=' * 65)
print('MODEL COMPARISON (Test Set: 2024 Season)')
print('=' * 65)
display(metrics_df.round(4))

# Highlight best model
best_f1_idx = metrics_df['F1 Score'].idxmax()
best_auc_idx = metrics_df['ROC-AUC'].idxmax()
print(f'\nBest F1 Score: {metrics_df.loc[best_f1_idx, "Model"]} ({metrics_df.loc[best_f1_idx, "F1 Score"]:.4f})')
print(f'Best ROC-AUC: {metrics_df.loc[best_auc_idx, "Model"]} ({metrics_df.loc[best_auc_idx, "ROC-AUC"]:.4f})')

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Metrics comparison
ax1 = axes[0]
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
x = np.arange(len(metrics_to_plot))
width = 0.25

models = ['Logistic Regression', 'Random Forest', 'XGBoost']
colors_list = ['#3498db', '#2ecc71', '#e74c3c']

for i, (model_name, color) in enumerate(zip(models, colors_list)):
    values = [metrics_df.loc[metrics_df['Model'] == model_name, m].values[0] for m in metrics_to_plot]
    ax1.bar(x + i * width, values, width, label=model_name, color=color, alpha=0.85, edgecolor='white')

ax1.set_xlabel('Metric', fontsize=12)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax1.set_xticks(x + width)
ax1.set_xticklabels(metrics_to_plot, rotation=45, ha='right')
ax1.legend(fontsize=10)
ax1.set_ylim(0, 1.1)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# CV scores comparison
ax2 = axes[1]
cv_scores = {
    'Logistic Regression': lr_cv_scores,
    'Random Forest': rf_cv_scores,
    'XGBoost': xgb_cv_scores
}

bp = ax2.boxplot(cv_scores.values(), label=cv_scores.keys(), patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], colors_list):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax2.set_ylabel('ROC-AUC Score', fontsize=12)
ax2.set_title('Cross-Validation ROC-AUC Comparison', fontsize=14, fontweight='bold')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/figures/08_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Best Model Evaluation

We select the best-performing model based on ROC-AUC and F1 Score and perform detailed evaluation.


In [ ]:
# Select best model (XGBoost typically performs best)
best_model_name = metrics_df.loc[metrics_df['ROC-AUC'].idxmax(), 'Model']
print(f'Best model: {best_model_name}')

if best_model_name == 'XGBoost':
    best_model = xgb_model
    best_pred = xgb_pred
    best_proba = xgb_proba
elif best_model_name == 'Random Forest':
    best_model = rf_model
    best_pred = rf_pred
    best_proba = rf_proba
else:
    best_model = lr_model
    best_pred = lr_pred
    best_proba = lr_proba

### 6.1 Confusion Matrix & Classification Report

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
ax1 = axes[0]
cm = confusion_matrix(y_test, best_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Non-Podium', 'Podium'],
            yticklabels=['Non-Podium', 'Podium'],
            annot_kws={'fontsize': 14})
ax1.set_xlabel('Predicted', fontsize=12)
ax1.set_ylabel('Actual', fontsize=12)
ax1.set_title(f'Confusion Matrix ({best_model_name})', fontsize=14, fontweight='bold')

# Normalized confusion matrix
ax2 = axes[1]
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=ax2,
            xticklabels=['Non-Podium', 'Podium'],
            yticklabels=['Non-Podium', 'Podium'],
            annot_kws={'fontsize': 14})
ax2.set_xlabel('Predicted', fontsize=12)
ax2.set_ylabel('Actual', fontsize=12)
ax2.set_title(f'Normalized Confusion Matrix ({best_model_name})', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/09_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Classification report
print('\nClassification Report:')
print(classification_report(y_test, best_pred, target_names=['Non-Podium', 'Podium']))

### 6.2 ROC Curve Comparison (All Models)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# All three models
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_proba)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_proba)

ax.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {lr_metrics["ROC-AUC"]:.3f})', 
        color='#3498db', linewidth=2)
ax.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {rf_metrics["ROC-AUC"]:.3f})', 
        color='#2ecc71', linewidth=2)
ax.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC = {xgb_metrics["ROC-AUC"]:.3f})', 
        color='#e74c3c', linewidth=2.5)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random Classifier')
ax.fill_between(fpr_xgb, tpr_xgb, alpha=0.1, color='#e74c3c')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/figures/10_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 Feature Importance

Using XGBoost's built-in feature importance to understand what drives podium predictions.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Get feature importances from XGBoost
importances = xgb_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=True)

colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(feature_importance_df)))
ax.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], 
        color=colors, edgecolor='white', linewidth=0.5)

for i, (val, name) in enumerate(zip(feature_importance_df['Importance'], feature_importance_df['Feature'])):
    ax.text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('Feature Importance (Gain)', fontsize=12)
ax.set_title('XGBoost Feature Importance', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/figures/11_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 Most Important Features:')
for _, row in feature_importance_df.tail(5).iterrows():
    print(f'  {row["Feature"]}: {row["Importance"]:.4f}')

---
## 7. Key Insights

### 7.1 What Factors Matter Most for Podium Finishes?

Based on our feature importance analysis and EDA, we can identify the key predictors.

In [ ]:
print('=== KEY DRIVERS OF PODIUM FINISHES ===')
print('\n1. GRID POSITION')
for gp in range(1, 6):
    rate = df[df['grid_position'] == gp]['podium_finish'].mean()
    print(f'   P{gp} start -> {rate:.1%} podium rate')

print(f'   P11+ start -> {df[df["grid_position"] > 10]["podium_finish"].mean():.1%} podium rate')

print('\n2. CONSTRUCTOR QUALITY')
for tier in ['top', 'mid', 'backmarker']:
    rate = df[df['constructor_tier'] == tier]['podium_finish'].mean()
    print(f'   {tier.capitalize()} team -> {rate:.1%} podium rate')

print('\n3. DRIVER EXPERIENCE')
exp_median = df['driver_experience'].median()
experienced = df[df['driver_experience'] >= exp_median]['podium_finish'].mean()
less_exp = df[df['driver_experience'] < exp_median]['podium_finish'].mean()
print(f'   High experience (>= {exp_median}): {experienced:.1%} podium rate')
print(f'   Low experience (< {exp_median}): {less_exp:.1%} podium rate')

### 7.2 Constructor Advantage Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Constructor podium rate by season
top_constructors = ['Red Bull Racing', 'Mercedes', 'Ferrari', 'McLaren']

x_pos = np.arange(len([2020, 2021, 2022, 2023, 2024]))
width = 0.2
colors = ['#1E41FF', '#00D2BE', '#DC0000', '#FF8000']

for i, (team, color) in enumerate(zip(top_constructors, colors)):
    rates = []
    for season in [2020, 2021, 2022, 2023, 2024]:
        team_season = df[(df['constructor'] == team) & (df['season'] == season)]
        if len(team_season) > 0:
            rates.append(team_season['podium_finish'].mean() * 100)
        else:
            rates.append(0)
    ax.bar(x_pos + i * width, rates, width, label=team, color=color, alpha=0.85)

ax.set_xlabel('Season', fontsize=12)
ax.set_ylabel('Podium Rate (%)', fontsize=12)
ax.set_title('Podium Rate by Constructor & Season', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos + width * 1.5)
ax.set_xticklabels([2020, 2021, 2022, 2023, 2024])
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/figures/12_constructor_advantage.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.3 Circuit Type Effects

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Street vs permanent - podium rate by grid position
ax1 = axes[0]
for circ_type in ['permanent', 'street']:
    subset = df[df['circuit_type'] == circ_type]
    rates = subset.groupby('grid_position')['podium_finish'].mean()
    rates = rates[rates.index <= 10]
    ax1.plot(rates.index, rates.values * 100, marker='o', linewidth=2, 
             label=f'{circ_type.capitalize()}', markersize=6)

ax1.set_xlabel('Grid Position', fontsize=12)
ax1.set_ylabel('Podium Rate (%)', fontsize=12)
ax1.set_title('Podium Rate by Grid Position & Circuit Type', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.invert_xaxis()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Street circuits have less overtaking
ax2 = axes[1]
street_data = df[df['circuit_type'] == 'street']
perm_data = df[df['circuit_type'] == 'permanent']

# Position change (grid to finish) for podium finishers
street_podium = street_data[street_data['podium_finish'] == 1]
perm_podium = perm_data[perm_data['podium_finish'] == 1]

street_pos_change = (street_podium['grid_position'] - street_podium['final_position']).mean()
perm_pos_change = (perm_podium['grid_position'] - perm_podium['final_position']).mean()

bars = ax2.bar(['Street Circuits', 'Permanent Circuits'], 
               [abs(street_pos_change), abs(perm_pos_change)],
               color=['#e67e22', '#3498db'], edgecolor='white', linewidth=1.5, width=0.5)
ax2.set_ylabel('Avg. Positions Gained (Podium Finishers)', fontsize=11)
ax2.set_title('Overtaking Difficulty by Circuit Type', fontsize=13, fontweight='bold')

for bar, val in zip(bars, [abs(street_pos_change), abs(perm_pos_change)]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.1f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/13_circuit_type_effects.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.4 Top Performers by Era

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 2020-2021 era
ax1 = axes[0]
era1 = df[(df['season'].isin([2020, 2021])) & (df['podium_finish'] == 1)]
era1_top = era1['driver'].value_counts().head(8)

colors1 = plt.cm.Blues(np.linspace(0.4, 0.9, len(era1_top)))[::-1]
ax1.barh(era1_top.index[::-1], era1_top.values[::-1], color=colors1[::-1])
ax1.set_title('Top Podium Finishers (2020-2021)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Podium Finishes', fontsize=11)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# 2022-2024 era
ax2 = axes[1]
era2 = df[(df['season'].isin([2022, 2023, 2024])) & (df['podium_finish'] == 1)]
era2_top = era2['driver'].value_counts().head(8)

colors2 = plt.cm.Oranges(np.linspace(0.4, 0.9, len(era2_top)))[::-1]
ax2.barh(era2_top.index[::-1], era2_top.values[::-1], color=colors2[::-1])
ax2.set_title('Top Podium Finishers (2022-2024)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Podium Finishes', fontsize=11)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/figures/14_top_performers_era.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Conclusions

### Key Findings

1. **Grid position is the strongest predictor of podium finishes.** Starting from the front row (P1-P2) gives roughly a 60-70% chance of finishing on the podium, while starting outside the top 6 drops the probability below 20%.

2. **Constructor quality dominates driver-level factors.** Top-tier teams (Red Bull, Mercedes, Ferrari, McLaren) account for over 95% of all podium finishes. The car matters more than the driver in modern F1.

3. **Circuit type affects overtaking but not overall podium probability.** Street circuits like Monaco, Singapore, and Baku see less position changes from grid to finish, making qualifying even more critical.

4. **Team dominance shifts over eras.** Mercedes dominated 2020, Red Bull/Verstappen controlled 2021-2023, while McLaren emerged as a strong contender in 2024.

### Model Performance Summary

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|-------|----------|-----------|--------|----|----|


In [ ]:
# Print final summary
print('=' * 70)
print('FINAL MODEL PERFORMANCE SUMMARY')
print('=' * 70)
print(f'\nDataset: 103 races, 2140 entries (2020-2024 F1 Seasons)')
print(f'Train set: 2020-2023 ({X_train.shape[0]} entries)')
print(f'Test set: 2024 ({X_test.shape[0]} entries)')
print(f'\nBest Model: {best_model_name}')
print(f'Best ROC-AUC: {metrics_df.loc[best_f1_idx, "ROC-AUC"]:.4f}')
print(f'Best F1 Score: {metrics_df.loc[best_f1_idx, "F1 Score"]:.4f}')
print('\n')
display(metrics_df.round(4))

### Model Limitations

1. **Simulated data**: While designed to reflect real F1 patterns, this is synthetic data. Real-world F1 data would include many more features (pit strategy, tire degradation, safety car periods, etc.).
2. **No temporal sequencing**: The model treats each race independently without considering race-to-race momentum or championship context.
3. **Limited features**: Real F1 prediction would benefit from telemetry data, practice/qualifying session times, and pit stop data.
4. **Class imbalance**: Despite using class weights, predicting rare events (podium finishes at ~15%) remains challenging.

### Future Improvements

- Incorporate real F1 data from the Ergast API or FastF1 library
- Add temporal features (form over last N races, championship standings)
- Implement multi-class prediction (predicting exact position or top-3 with probabilities)
- Use neural networks or ensemble stacking for improved performance
- Add weather forecast data as a pre-race feature
- Include pit stop strategy optimization as a secondary model

---

**Author**: F1 Data Science Project | **Last Updated**: 2025

*This project was created for educational and demonstration purposes.*